---
short_title: Statistiken
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# Statistiken berechnen

```{caution} Work In Progress
Diese Inhalte sind noch in Bearbeitung.
```

Nach der Bereinigung beantworten wir typische Fragen aus dem Bibliotheksalltag:

* Wie viele Medien sind im Bestand und wie häufig wurden sie ausgeliehen?
* Welcher Standort ist besonders stark frequentiert?
* Gibt es Unterschiede zwischen den Medienarten?

Die Bereinigung aus dem [vorigen Kapitel](020-Datenbereinigen.ipynb) fassen wir
in der Funktion `bereinige()` zusammen.

In [1]:
import pandas as pd

```{note}
**Hinweis zur Browser-Version:** Der Python-Kernel von JupyterLite (über den
Start-Knopf der Seite) läuft im Browser und hat ein anderes Environment als
Ihre lokale Installation: Die Beispieldatei
`assets/data/bibliothek_unsauber.xlsx` liegt dort nicht im Projekt, und die
Excel-Engine `openpyxl` fehlt. Die folgende Zelle lädt beides **nur im
Browser** von der Kurs-Website nach (erkannt an
`sys.platform == "emscripten"`). Lokal in JupyterLab tut sie nichts.
```

In [ ]:
import sys

# Pyodide (JupyterLite) meldet sich mit der Plattform "emscripten";
# lokal (JupyterLab, Build) ist sie z. B. "darwin" oder "linux".
if sys.platform == "emscripten":
    from pathlib import Path

    from js import location

    # 1) Excel-Engine openpyxl fehlt in JupyterLite. Die Wheels liegen auf
    #    der Kurs-Website (Ordner assets/pypi) und werden direkt installiert.
    import micropip

    _wheels = (
        "et_xmlfile-2.0.0-py3-none-any.whl",
        "openpyxl-3.1.5-py2.py3-none-any.whl",
    )
    _fehler = None
    for _basis in (f"{location.origin}/pypi", "pypi"):
        try:
            await micropip.install(
                [f"{_basis}/{_w}" for _w in _wheels], keep_going=True
            )
            _fehler = None
            break
        except Exception as _e:  # noqa: BLE001
            _fehler = _e
    if _fehler is not None:
        raise RuntimeError(f"openpyxl konnte nicht installiert werden: {_fehler}")

    # 2) Beispieldatei von der Kurs-Website ins virtuelle Dateisystem laden,
    #    sodass relative Pfade wie "../assets/data/..." funktionieren
    def stelle_daten_bereit(pfad):
        """Lädt eine Beispieldatei von der Kurs-Website ins virtuelle Dateisystem."""
        ziel = Path(pfad)
        if ziel.exists():
            return ziel

        from js import XMLHttpRequest

        for url in (f"{location.origin}/{ziel.name}", ziel.name):
            anfrage = XMLHttpRequest.new()
            # synchron möglich, weil der Kernel in einem Web Worker läuft
            anfrage.open("GET", url, False)
            try:
                anfrage.responseType = "arraybuffer"
                anfrage.send()
            except Exception:
                continue  # nächste Kandidaten-URL versuchen
            if anfrage.status == 200:
                inhalt = anfrage.response.to_py()
                if not isinstance(inhalt, str):  # ArrayBuffer -> Bytes
                    ziel.parent.mkdir(parents=True, exist_ok=True)
                    ziel.write_bytes(bytes(inhalt))
                    return ziel
        raise FileNotFoundError(
            f"{pfad} weder lokal noch auf der Kurs-Website gefunden."
        )

    stelle_daten_bereit("../assets/data/bibliothek_unsauber.xlsx")

    # 3) Ausgabeverzeichnis für die Abbildungen sicherstellen
    Path("../assets/090").mkdir(parents=True, exist_ok=True)

In [2]:
STANDORT_MAPPING = {
    "ZB": "Zentralbibliothek",
    "zentralbibliothek": "Zentralbibliothek",
    "Nord": "Zweigstelle Nord",
    "Zweigstelle Nord": "Zweigstelle Nord",
    "Süd": "Zweigstelle Süd",
    "Zweigstelle Sued": "Zweigstelle Süd",
    "Zweigstelle Süd": "Zweigstelle Süd",
    "West": "Campus West",
    "campus west": "Campus West",
    "Campus West": "Campus West",
    "mag": "Magazin",
    "MAG": "Magazin",
    "Magazin": "Magazin",
    "Zentralbibliothek": "Zentralbibliothek",
}

In [3]:
MEDIUM_MAPPING = {
    "buch": "Buch",
    "Buch": "Buch",
    "eBook": "E-Book",
    "E-Book": "E-Book",
    "zeitschrift": "Zeitschrift",
    "Zeitschr.": "Zeitschrift",
    "Zeitschrift": "Zeitschrift",
    "dvd": "DVD",
    "DVD": "DVD",
    "hoerbuch": "Hörbuch",
    "Hörbuch": "Hörbuch",
}

In [4]:
def parse_deutsche_zahl(serie):
    """Wandelt Text mit deutschem Tausenderpunkt in Zahlen um."""
    text = serie.astype("string").str.strip()
    hat_tausenderpunkt = text.str.match(r"^-?\d{1,3}(\.\d{3})+$", na=False)
    text = text.where(~hat_tausenderpunkt, text.str.replace(".", "", regex=False))
    return pd.to_numeric(text, errors="coerce")

In [5]:
def bereinige(pfad):
    """Liest die unsaubere Excel-Datei ein und gibt saubere Daten zurueck."""
    df = pd.read_excel(pfad, sheet_name="Ausleihen")
    df = df.drop_duplicates().reset_index(drop=True)

    df["Standort"] = df["Standort"].str.strip().map(STANDORT_MAPPING)
    df["Medium"] = df["Medium"].str.strip().map(MEDIUM_MAPPING)

    df["Erscheinungsjahr"] = pd.to_numeric(
        df["Erscheinungsjahr"].astype("string").str.strip(), errors="coerce"
    )
    df["Ausleihzahlen"] = parse_deutsche_zahl(df["Ausleihzahlen"])
    df["Zugangsdatum"] = pd.to_datetime(
        df["Zugangsdatum"], format="mixed", dayfirst=True, errors="coerce"
    )

    df = df.rename(columns={
        "Titel": "titel",
        "Autor*in": "autor",
        "Erscheinungsjahr": "jahr",
        "Ausleihzahlen": "ausleihen",
        "Standort": "standort",
        "Medium": "medium",
        "Zugangsdatum": "zugang",
    })
    return df.dropna(subset=["titel", "autor"]).reset_index(drop=True)

In [6]:
bibliothek = bereinige("../assets/data/bibliothek_unsauber.xlsx")
bibliothek.head()

,titel,autor,jahr,ausleihen,standort,medium,zugang
0,Metadaten in der Praxis,"Behrens, Carl",2007,8,Magazin,DVD,2007-03-23
1,Grundlagen der Katalogisierung,"Dittrich, Erik",<NA>,186,Campus West,Zeitschrift,2013-04-22
2,Digitalisierung im Archiv,"Engel, Friederike",1993,21,Magazin,E-Book,1993-06-13
3,Open Access verstehen,"Fischer, Gerd",2024,<NA>,Zweigstelle Süd,E-Book,2024-07-19
4,Bibliotheken im digitalen Wandel,"Hoffmann, Ingo",2006,395,Campus West,E-Book,2006-03-17


## Überblick mit `describe()`

`describe()` berechnet Kennzahlen wie Mittelwert, Median und Quartile für
numerische Spalten.

In [7]:
bibliothek[["jahr", "ausleihen"]].describe()

,jahr,ausleihen
count,42.0,48.0
mean,2008.714286,848.4375
std,10.114674,2059.197281
min,1990.0,5.0
25%,2005.0,119.25
50%,2008.0,249.0
75%,2016.0,320.0
max,2024.0,8798.0


## Häufigkeiten mit `value_counts()`

`value_counts()` zählt, wie häufig jeder Wert vorkommt – ideal für Kategorien
wie Standort oder Medium.

In [8]:
bibliothek["medium"].value_counts()

medium
E-Book         13
Zeitschrift    11
DVD            10
Buch           10
Hörbuch         9
Name: count, dtype: int64

In [9]:
bibliothek["standort"].value_counts()

standort
Magazin              12
Zweigstelle Süd      12
Zweigstelle Nord     10
Zentralbibliothek    10
Campus West           9
Name: count, dtype: int64

## Gruppieren und aggregieren

{term}`Pandas` `groupby()` fasst Zeilen mit gleichem Wert zusammen. Mit `agg([...])` fordern
Sie mehrere Kennzahlen gleichzeitig an.

In [10]:
bibliothek.groupby("standort")["ausleihen"].agg(["count", "sum", "mean", "median"]).round(1)

,count,sum,mean,median
standort,,,,
Campus West,9,2541,282.3,271.0
Magazin,11,14330,1302.7,129.0
Zentralbibliothek,9,4199,466.6,122.0
Zweigstelle Nord,9,8828,980.9,243.0
Zweigstelle Süd,10,10827,1082.7,276.5


In [11]:
bibliothek.groupby("medium")["ausleihen"].agg(["count", "sum", "mean"]).round(1)

,count,sum,mean
medium,,,
Buch,9,18968,2107.6
DVD,9,1351,150.1
E-Book,11,5463,496.6
Hörbuch,8,12434,1554.2
Zeitschrift,11,2509,228.1


## Zwei Merkmale kombinieren

Eine `pivot_table` kreuzt zwei Kategorien – hier Standort und Medium – und
aggregiert die Ausleihzahlen.

In [12]:
bibliothek.pivot_table(
    index="standort",
    columns="medium",
    values="ausleihen",
    aggfunc="sum",
)

medium,Buch,DVD,E-Book,Hörbuch,Zeitschrift
standort,,,,,
Campus West,348,597,918,240,438
Magazin,8862,65,605,4511,287
Zentralbibliothek,165,134,3382,467,51
Zweigstelle Nord,296,147,558,7105,722
Zweigstelle Süd,9297,408,0,111,1011


## Die ausleihstärksten Titel

In [13]:
bibliothek.nlargest(5, "ausleihen")[["titel", "standort", "medium", "ausleihen"]]

,titel,standort,medium,ausleihen
27,Records Management,Zweigstelle Süd,Buch,8798
5,Literaturrecherche effizient,Magazin,Buch,8611
45,Digitalisierung im Archiv,Zweigstelle Nord,Hörbuch,7100
26,Literaturrecherche effizient,Magazin,Hörbuch,4511
35,Leseförderung und Medienpädagogik,Zentralbibliothek,E-Book,3033


## Ausleihen im Zeitverlauf

Gruppiert nach Erscheinungsjahr lässt sich ein Trend erkennen. Solche
Zeitreihen eignen sich gut für ein Liniendiagramm im
[nächsten Kapitel](040-Visualisierung.ipynb).

In [14]:
nach_jahr = bibliothek.groupby("jahr")["ausleihen"].sum()
nach_jahr

jahr
1990     255
1991    4860
1993      21
1994     466
1996     111
1998     346
2000     566
2005     890
2006     485
2007     548
2008     156
2009    3334
2011     344
2013     327
2014     129
2015       5
2016    7691
2019     240
2020     247
2021      23
2023      12
2024    9432
Name: ausleihen, dtype: Int64

## Übung

````{exercise}
:label: statistik-uebung

1. Berechnen Sie die durchschnittlichen Ausleihzahlen je Medium **und**
   Standort mit `groupby([...])`.
2. Ermitteln Sie den Standort mit den meisten Ausleihen (`idxmax()`).
3. Zählen Sie, wie viele verschiedene Titel (nicht Ausleihen) es je Medium
   gibt.

````